# 4부 자습 노트북 — AdaBoost

본 노트북은 *직접 실행하며* 학습하는 자료이다. 4부 이론 교재(`part4_adaboost_HARD_이론.md`)의 핵심 코드를 모두 돌려볼 수 있다.

**시리즈에서 본 부의 위치**: 3부(RF)에 이어 부스팅의 *첫 사례*인 AdaBoost를 다룬다. *왜 캘리포니아에서 AdaBoost가 처참하게 실패하는지*가 본 부의 핵심 사건이다.

**데이터셋**:
- **Ames** (2,930 × 82) — AdaBoost가 잘 작동하는 사례
- **California** (20,640 × 9) — AdaBoost가 실패하는 사례 (capped 이상치)

## 환경 준비와 데이터 로딩

In [ ]:
# Colab 등에서 처음 한 번만 실행
# !pip install koreanize-matplotlib --quiet

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import koreanize_matplotlib
import warnings
warnings.filterwarnings("ignore")

plt.rcParams["axes.unicode_minus"] = False

URL_AMES  = "https://raw.githubusercontent.com/leina99-lab/classes/main/AI%ED%94%84%EB%A1%9C%EA%B7%B8%EB%9E%98%EB%B0%8D/data/AmesHousing.csv"
URL_CALIF = "https://raw.githubusercontent.com/ageron/handson-ml2/master/datasets/housing/housing.csv"

try:
    ames_raw  = pd.read_csv(URL_AMES)
    calif_raw = pd.read_csv(URL_CALIF)
except Exception:
    rng = np.random.default_rng(42)
    n = 2900
    ames_raw = pd.DataFrame({
        "Overall Qual": rng.integers(1, 11, n),
        "Gr Liv Area":  rng.integers(500, 4500, n),
        "Year Built":   rng.integers(1900, 2010, n),
    })
    ames_raw["SalePrice"] = (
        50000 + ames_raw["Overall Qual"] * 25000
        + ames_raw["Gr Liv Area"] * 60 + rng.normal(0, 20000, n)
    ).astype(int)
    
    n2 = 5000
    calif_raw = pd.DataFrame({
        "longitude":      rng.uniform(-124, -114, n2),
        "latitude":       rng.uniform(32, 42, n2),
        "housing_median_age": rng.integers(1, 52, n2),
        "total_rooms":    rng.integers(2, 6000, n2),
        "population":     rng.integers(3, 5000, n2),
        "households":     rng.integers(2, 1000, n2),
        "median_income":  rng.uniform(0.5, 15, n2),
        "median_house_value": np.clip(rng.normal(200000, 100000, n2), 15000, 500001).astype(int)
    })

print(f"Ames:       {ames_raw.shape}")
print(f"California: {calif_raw.shape}")

In [ ]:
def prepare_ames(df_in):
    df = df_in.copy()
    df = df.drop(columns=[c for c in ["Order", "PID"] if c in df.columns])
    for c in ["Pool QC", "Misc Feature", "Alley", "Fence", "Fireplace Qu",
              "Garage Qual", "Garage Cond", "Bsmt Qual", "Bsmt Cond"]:
        if c in df.columns:
            df[c] = df[c].fillna("None")
    num = df.select_dtypes("number").columns
    df[num] = df[num].fillna(df[num].median())
    df = df[df["Gr Liv Area"] < 4000].copy()
    if all(c in df.columns for c in ["1st Flr SF", "2nd Flr SF", "Total Bsmt SF"]):
        df["Total SF"] = df["1st Flr SF"] + df["2nd Flr SF"] + df["Total Bsmt SF"]
    return df

ames = prepare_ames(ames_raw)
y_ames = np.log1p(ames["SalePrice"])
X_ames = ames.select_dtypes("number").drop(columns=["SalePrice"])

# California 전처리
calif = calif_raw.copy()
if "ocean_proximity" in calif.columns:
    calif = pd.get_dummies(calif, columns=["ocean_proximity"], drop_first=True)
num = calif.select_dtypes("number").columns
calif[num] = calif[num].fillna(calif[num].median())
y_calif = calif["median_house_value"]
X_calif = calif.drop(columns=["median_house_value"])

print(f"X_ames:  {X_ames.shape}")
print(f"X_calif: {X_calif.shape}")

---
## 0장 분산 감소에서 편향 감소로 — 부스팅의 정신

3부 RF는 *분산을 줄이는* 앙상블이었다. AdaBoost를 비롯한 *부스팅*은 *편향을 줄이는* 앙상블이다.

| 측면 | 배깅 (RF) | 부스팅 (AdaBoost, GBM) |
|---|---|---|
| 학습 방향 | 병렬 | 순차 |
| 트리 의존성 | 독립 | 이전 트리에 의존 |
| 줄이는 것 | 분산 | 편향 |

---
## 1장 가중치 갱신 — 다음 트리가 무엇을 보고 학습하나

AdaBoost는 *틀린 샘플의 가중치를 키운다*. 다음 트리는 *어려운 샘플에 집중*해 학습한다.

In [ ]:
# 가중치 갱신 직접 시뮬레이션 (이진 분류 예시)
import numpy as np
np.random.seed(42)

n = 10
y_true = np.array([1, 1, -1, 1, -1, -1, 1, 1, -1, 1])
y_pred = np.array([1, -1, -1, 1, 1, -1, 1, 1, -1, -1])  # 3개 틀림 (인덱스 1, 4, 9)

# 초기 가중치 — 모두 1/n
w = np.ones(n) / n
print(f"초기 가중치: {w.round(3)}")

# 에러율 계산
incorrect = (y_pred != y_true)
err = np.sum(w[incorrect]) / np.sum(w)
print(f"\n에러율: {err:.4f}")

# 알파 (트리 가중치)
alpha = 0.5 * np.log((1 - err) / err)
print(f"트리 가중치 α: {alpha:.4f}")

# 가중치 갱신: 틀린 샘플은 키우고, 맞은 샘플은 줄임
w_new = w * np.exp(-alpha * y_true * y_pred)
w_new /= w_new.sum()  # 정규화

print(f"\n갱신 후 가중치: {w_new.round(3)}")
print(f"\n→ 틀린 샘플(인덱스 1, 4, 9)의 가중치가 다른 샘플보다 크다.")

---
## 2장 약학습기의 누적 — 결정 경계가 어떻게 정교해지나

여러 *얕은 트리* (약학습기)를 *가중합*하여 점점 정교한 모델을 만든다.

In [ ]:
from sklearn.ensemble import AdaBoostRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import cross_val_score

# 라운드 수에 따른 R² 변화
print(f"{'n_est':>8s}  {'CV R²':>10s}")
print("-" * 22)
for n_est in [1, 5, 10, 25, 50, 100]:
    ab = AdaBoostRegressor(
        estimator=DecisionTreeRegressor(max_depth=3),
        n_estimators=n_est,
        random_state=42
    )
    r2 = cross_val_score(ab, X_ames, y_ames, cv=3, scoring="r2", n_jobs=-1).mean()
    print(f"{n_est:>8d}  {r2:>10.4f}")

---
## 3장 AdaBoost.R2 — 회귀로의 확장

분류용 AdaBoost.M1을 회귀로 확장한 변종. sklearn의 `AdaBoostRegressor`가 이를 구현.

**손실함수 옵션**: `linear`, `square`, `exponential`. 기본은 `linear`.

In [ ]:
print(f"{'loss':>15s}  {'CV R²':>10s}")
print("-" * 28)
for loss in ["linear", "square", "exponential"]:
    ab = AdaBoostRegressor(
        estimator=DecisionTreeRegressor(max_depth=3),
        n_estimators=100,
        loss=loss,
        random_state=42
    )
    r2 = cross_val_score(ab, X_ames, y_ames, cv=3, scoring="r2", n_jobs=-1).mean()
    print(f"{loss:>15s}  {r2:>10.4f}")

---
## 4장 AdaBoost.M1과 AdaBoost.R2 — 두 변종

M1은 *분류용*, R2는 *회귀용*. sklearn은 분류에 `AdaBoostClassifier`, 회귀에 `AdaBoostRegressor`를 제공한다.

In [ ]:
from sklearn.ensemble import AdaBoostClassifier
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import cross_val_score

# 분류 예시 (유방암 데이터)
data = load_breast_cancer()
X_clf, y_clf = data.data, data.target

print("=== 분류 AdaBoost.M1 ===")
ab_clf = AdaBoostClassifier(n_estimators=100, random_state=42)
acc = cross_val_score(ab_clf, X_clf, y_clf, cv=5, scoring="accuracy").mean()
print(f"  정확도: {acc:.4f}")

print("\n=== 회귀 AdaBoost.R2 (Ames) ===")
ab_reg = AdaBoostRegressor(
    estimator=DecisionTreeRegressor(max_depth=3),
    n_estimators=100, random_state=42
)
r2 = cross_val_score(ab_reg, X_ames, y_ames, cv=3, scoring="r2", n_jobs=-1).mean()
print(f"  R²: {r2:.4f}")

---
## 5장 AdaBoost의 핵심 매개변수

`learning_rate`와 `n_estimators`의 트레이드오프, `estimator` (약학습기 깊이)의 효과.

In [ ]:
# learning_rate 효과
print(f"{'learning_rate':>15s}  {'CV R²':>10s}")
print("-" * 28)
for lr in [0.1, 0.5, 1.0, 2.0]:
    ab = AdaBoostRegressor(
        estimator=DecisionTreeRegressor(max_depth=3),
        n_estimators=100,
        learning_rate=lr,
        random_state=42
    )
    r2 = cross_val_score(ab, X_ames, y_ames, cv=3, scoring="r2", n_jobs=-1).mean()
    print(f"{lr:>15.2f}  {r2:>10.4f}")

In [ ]:
# 약학습기 깊이 효과
print(f"{'max_depth':>10s}  {'CV R²':>10s}")
print("-" * 22)
for d in [1, 3, 5, 7]:
    ab = AdaBoostRegressor(
        estimator=DecisionTreeRegressor(max_depth=d),
        n_estimators=100, random_state=42
    )
    r2 = cross_val_score(ab, X_ames, y_ames, cv=3, scoring="r2", n_jobs=-1).mean()
    print(f"{d:>10d}  {r2:>10.4f}")

---
## 6장 Ames에서의 AdaBoost — 잘 작동하는 사례

정제된 데이터에서는 AdaBoost가 *튜닝한 단일 트리*와 비슷한 수준의 성능을 낸다.

In [ ]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, AdaBoostRegressor

print(f"{'모델':<30s}  {'R²':>10s}")
print("-" * 42)
for name, m in [
    ("단일 트리 (기본)",      DecisionTreeRegressor(random_state=42)),
    ("단일 트리 (튜닝)",      DecisionTreeRegressor(max_depth=7, min_samples_leaf=10, random_state=42)),
    ("AdaBoost",              AdaBoostRegressor(
        estimator=DecisionTreeRegressor(max_depth=3),
        n_estimators=100, random_state=42)),
    ("랜덤 포레스트",         RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)),
]:
    r2 = cross_val_score(m, X_ames, y_ames, cv=5, scoring="r2", n_jobs=-1).mean()
    print(f"{name:<30s}  {r2:>10.4f}")

print("\n→ AdaBoost가 Ames에서는 단일 트리 수준. 앙상블 이득이 크지 않음.")

---
## 7장 캘리포니아에서의 실패 — AdaBoost의 약점

캘리포니아 데이터의 *capped 이상치 \$500,001*이 AdaBoost의 가중치를 *폭발*시킨다. 이게 본 부의 *핵심 사건*이다.

In [ ]:
# 캘리포니아의 capped 이상치 확인
print(f"가격 = $500,001인 샘플: {(y_calif == 500001).sum()}건")
print(f"전체에서 비율:           {(y_calif == 500001).mean()*100:.2f}%")
print(f"\n가격 분포 상위 10:")
print(y_calif.value_counts().head(10))

# 히스토그램
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(y_calif, bins=80, color="#1F3A5F", alpha=0.7)
ax.axvline(500001, color="#C0392B", linestyle="--", linewidth=2, label="capped = $500,001")
ax.set_xlabel("median_house_value")
ax.set_ylabel("빈도")
ax.set_title("캘리포니아 가격 분포 — 오른쪽 끝의 capped 봉우리")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# AdaBoost의 실패 재현
ab_calif = AdaBoostRegressor(
    estimator=DecisionTreeRegressor(max_depth=3),
    n_estimators=100, random_state=42
)
r2_ab = cross_val_score(ab_calif, X_calif, y_calif, cv=3, scoring="r2", n_jobs=-1).mean()
print(f"AdaBoost 캘리포니아 R²: {r2_ab:.4f}")
print(f"\n→ R² 0.2 근처 — 처참한 실패!")
print(f"→ 다음 부 GBM의 Huber 손실로 0.65까지 회복한다.")

---
## 8장 GBM으로의 다리 — 손실함수 일반화

AdaBoost는 *지수 손실 고정*. GBM은 *어떤 미분 가능 손실*도 가능. 손실 선택만으로 캘리포니아 R²가 0.23 → 0.65로 점프한다.

In [ ]:
from sklearn.ensemble import AdaBoostRegressor, GradientBoostingRegressor

print("=== 캘리포니아 데이터 ===")
print(f"{'모델':<30s}  {'R²':>10s}")
print("-" * 42)

ab = AdaBoostRegressor(n_estimators=100, random_state=42)
r2 = cross_val_score(ab, X_calif, y_calif, cv=3, scoring="r2", n_jobs=-1).mean()
print(f"{'AdaBoost (지수 손실)':<30s}  {r2:>10.4f}")

gbm_sq = GradientBoostingRegressor(loss="squared_error", n_estimators=100, random_state=42)
r2 = cross_val_score(gbm_sq, X_calif, y_calif, cv=3, scoring="r2", n_jobs=-1).mean()
print(f"{'GBM (제곱 손실)':<30s}  {r2:>10.4f}")

gbm_huber = GradientBoostingRegressor(loss="huber", n_estimators=100, random_state=42)
r2 = cross_val_score(gbm_huber, X_calif, y_calif, cv=3, scoring="r2", n_jobs=-1).mean()
print(f"{'GBM (Huber 손실)':<30s}  {r2:>10.4f}")

print("\n→ 손실함수 선택만으로 R²가 극적으로 변한다. 5부에서 자세히 본다.")

---
## 마무리

본 노트북에서 *직접 실행*한 AdaBoost의 핵심을 한 표로 정리한다.

| 장 | 핵심 도구 |
|---|---|
| 0장 부스팅 정신 | 분산 vs 편향 감소 |
| 1장 가중치 갱신 | np.exp(-α·y·ŷ), 정규화 |
| 2장 약학습기 누적 | n_estimators 효과 |
| 3장 AdaBoost.R2 | `loss="linear"`, `"square"`, `"exponential"` |
| 4장 M1 vs R2 | `AdaBoostClassifier`, `AdaBoostRegressor` |
| 5장 매개변수 | `learning_rate`, `n_estimators`, `estimator` |
| 6장 Ames 성공 | AdaBoost = 단일 트리 수준 |
| 7장 캘리포니아 실패 | capped 이상치 → R² 0.23 |
| 8장 GBM 다리 | 손실 일반화 → R² 0.65 회복 |

### 다음 단계

5부로 이동하여 **GBM**(`part5_GBM_HARD_자습노트북.ipynb`)을 자습한다. *손실함수 일반화*가 어떻게 캘리포니아를 회복시키는지 직접 확인한다.